# Does DAS add anything beyond the seismic network?

This notebook is the advisor-playable front end for the decisive SAFOD experiment. It starts from published exact-ID family labels, diagnoses the conventional-network baseline, and shows exactly what a DAS-only or joint pipeline would have to improve.

**Current answer: not demonstrated yet.** The data justify pursuing the test because neighboring published families are extremely similar on HRSN while the deep fiber measures a dense near-source wavefield. DAS gets credit only after a blinded same-interval comparison at matched false-discovery rate.

In [ ]:
# Play controls. Changing thresholds below is exploratory and cannot validate a claim.
from pathlib import Path
import sys

candidates = []
for base in [Path.cwd(), *Path.cwd().parents]:
    if (base / 'config' / 'pilot.json').exists() and base.name == 'repeaters_v2':
        candidates.append(base)
    nested = base / 'faultzone' / 'repeaters_v2'
    if (nested / 'config' / 'pilot.json').exists():
        candidates.append(nested)
PROJECT = next((path.resolve() for path in candidates), None)
if PROJECT is None:
    raise FileNotFoundError('Could not locate faultzone/repeaters_v2')
NOTEBOOK_ROOT = PROJECT.parents[1]
if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_ROOT))

EXPLORATORY_MIN_SCORE = 0.85
EXPLORATORY_MARGIN = 0.02
REBUILD_EXTERNAL_LABELS = False
REBUILD_HRSN_DIAGNOSTIC = False
print('Project:', PROJECT)

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image

OUT = PROJECT / 'outputs' / 'external_validation'
external_status = json.load(open(OUT / 'status.json'))
network_status = json.load(open(OUT / 'network_family_status.json'))
deep_status = json.load(open(PROJECT / 'outputs' / 'deep_das' / 'status.json'))
crosswalk = pd.read_csv(OUT / 'shortlist_external_crosswalk.csv')
population = pd.read_csv(OUT / 'published_validation_population.csv')
availability = pd.read_csv(OUT / 'network_waveform_availability.csv')
classifications = pd.read_csv(OUT / 'network_family_classification.csv')
scores = pd.read_csv(OUT / 'network_family_scores.csv')

headline = pd.DataFrame([
    {'result': 'Published target events', 'value': external_status['published_target_population_events'], 'claim_status': 'PASS population'},
    {'result': 'Published neighboring-family events', 'value': external_status['published_neighbor_population_events'], 'claim_status': 'PASS population'},
    {'result': 'Old single-anchor overmerges', 'value': external_status['discordant_single_anchor_overmerges'], 'claim_status': 'STOP old catalog'},
    {'result': 'Labeled events with HRSN windows', 'value': network_status['waveform_available_events'], 'claim_status': 'diagnostic only'},
    {'result': 'Events classified under frozen margin', 'value': network_status['classified_events'], 'claim_status': 'STOP classifier'},
    {'result': 'Blinded DAS-vs-network scans', 'value': 0, 'claim_status': 'NOT RUN'},
])
display(headline)

## 1. The external catalog repairs the labels

The Waldhauser--Schaff catalog combines waveform similarity, relative co-location, and similar event size. This matters here because correlation alone is high across several distinct nearby sequences. Matching is by exact event ID only; the location fields are retained as diagnostics, not used to transfer labels.

In [ ]:
matched = crosswalk[crosswalk['external_match_type'] == 'exact_event_id'].copy()
matched['cc'] = pd.to_numeric(matched['single_anchor_median_correlation'], errors='coerce')
display(matched[['event_id', 'ncedc_dd_origin_time', 'single_anchor_decision', 'cc', 'external_sequence_id', 'external_role', 'ncedc_minus_external_depth_km', 'diagnostic_outcome']])

sequence_order = sorted(matched['external_sequence_id'].dropna().unique())
xmap = {name: index for index, name in enumerate(sequence_order)}
role_colors = {'published_target_positive': 'tab:blue', 'published_neighbor_family_negative': 'tab:orange'}
fig, ax = plt.subplots(figsize=(10, 4.5), constrained_layout=True)
for role, group in matched.dropna(subset=['cc']).groupby('external_role'):
    x = np.array([xmap[value] for value in group['external_sequence_id']], dtype=float)
    jitter = np.linspace(-0.09, 0.09, len(group)) if len(group) > 1 else np.zeros(1)
    ax.scatter(x + jitter, group['cc'], s=60, color=role_colors.get(role, '0.5'), label=role, alpha=0.85)
    for xx, yy, event_id in zip(x + jitter, group['cc'], group['event_id']):
        ax.annotate(str(event_id), (xx, yy), xytext=(3, 3), textcoords='offset points', fontsize=7)
ax.axhline(0.85, color='tab:red', linestyle='--', linewidth=1, label='old single-anchor threshold')
ax.set_xticks(range(len(sequence_order)), sequence_order, rotation=20, ha='right')
ax.set_ylim(0.8, 1.005)
ax.set_ylabel('Correlation to the single 2026 seed')
ax.set_title('High single-seed similarity crosses published-family boundaries')
ax.legend(frameon=False, fontsize=8)
ax.grid(axis='y', alpha=0.2)
plt.show()

## 2. Frozen historical validation population and data availability

All 18 published events remain in the population. The present HRSN endpoint returns usable windows for ten: all six target events, two events in one neighboring family, and one event in each of two other neighboring families. Missing historical HRSN is an abstention, not a classification error or a negative waveform.

In [ ]:
population_view = population.groupby(['sequence_id', 'validation_role']).agg(
    published_events=('event_id', 'count'),
    shortlist_overlap=('in_frozen_shortlist', 'sum')
).reset_index()
availability_view = availability.groupby(['sequence_id', 'status']).size().unstack(fill_value=0).reset_index()
display(population_view.merge(availability_view, on='sequence_id', how='left'))
display(availability[availability['status'] != 'available'][['event_id', 'sequence_id', 'origin_time', 'error']])

## 3. HRSN correlation-only multi-anchor baseline

For each held-out event, the frozen diagnostic scores every published family by the median of up to three highest eligible pair correlations. It requires score at least 0.85 and a top-versus-second margin at least 0.02. All ten available events abstain because their family margins are smaller. Ignoring the abstention policy after seeing labels would yield only exploratory top-score accuracy, not a valid classifier result.

In [ ]:
display(pd.DataFrame([network_status]).drop(columns=['per_family'], errors='ignore'))
reason_counts = classifications.groupby(['waveform_available', 'classification_reason']).size().rename('events').reset_index()
display(reason_counts)
display(classifications[['event_id', 'published_sequence_id', 'waveform_available', 'top_sequence_id', 'top_family_score', 'second_family_score', 'classification_margin', 'predicted_sequence_id', 'classification_reason', 'correct']])

## 4. Why correlation is not enough, and what a stronger network baseline needs

The left panel compares pair correlations inside and across published families. The right panel adds the range of station-median residual lags after the catalog origin times are removed. Differential timing contains some separation, but also overlap. A final best-network baseline therefore needs event-held-out double-difference relocation rather than post-hoc correlation threshold changes.

In [ ]:
pair_summary = pd.read_csv(OUT / 'network_pair_summaries.csv')
pair_metrics = pd.read_csv(OUT / 'network_pair_metrics.csv')
pair_summary['same_family'] = pair_summary['same_published_family'].astype(str).str.lower().eq('true')
usable = pair_metrics[pair_metrics['usable'].astype(str).str.lower().eq('true')].copy()
lag_rows = []
for pair_name, group in usable.groupby('pair_name'):
    station_lags = group.groupby('station')['lag_s'].median().astype(float).to_numpy()
    lag_rows.append({
        'pair_name': pair_name,
        'station_lag_range_ms': 1000.0 * np.ptp(station_lags) if len(station_lags) else np.nan,
        'station_count': len(station_lags),
    })
features = pair_summary.merge(pd.DataFrame(lag_rows), on='pair_name', how='left')
summary_by_role = features.groupby('same_family')[['overall_median_correlation', 'weakest_band_median_correlation', 'station_lag_range_ms']].describe(percentiles=[0.1, 0.5, 0.9])
display(summary_by_role)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
for index, (same, group) in enumerate(features.groupby('same_family')):
    label = 'within published family' if same else 'between published families'
    color = 'tab:blue' if same else 'tab:orange'
    jitter = np.linspace(-0.12, 0.12, len(group))
    axes[0].scatter(np.full(len(group), index) + jitter, group['overall_median_correlation'], color=color, alpha=0.75, label=label)
    axes[1].scatter(group['station_lag_range_ms'], group['overall_median_correlation'], color=color, alpha=0.75, label=label)
axes[0].set_xticks([0, 1], ['between', 'within'])
axes[0].set_ylabel('Overall median HRSN correlation')
axes[0].set_title('Correlation distributions overlap')
axes[1].set_xlabel('Range of station-median residual lags (ms)')
axes[1].set_ylabel('Overall median HRSN correlation')
axes[1].set_title('Differential timing is useful but not sufficient here')
axes[1].legend(frameon=False, fontsize=8)
for ax in axes:
    ax.grid(alpha=0.2)
plt.show()

## 5. Advisor threshold sandbox — exploratory only

This cell lets an advisor see the abstention tradeoff. It reuses the already exposed labels, so its output must never be reported as held-out performance or used to overwrite the frozen configuration.

In [ ]:
sandbox = classifications[classifications['waveform_available'].astype(str).str.lower().eq('true')].copy()
sandbox['top_family_score'] = pd.to_numeric(sandbox['top_family_score'], errors='coerce')
sandbox['classification_margin'] = pd.to_numeric(sandbox['classification_margin'], errors='coerce')
accepted = (sandbox['top_family_score'] >= EXPLORATORY_MIN_SCORE) & (sandbox['classification_margin'] >= EXPLORATORY_MARGIN)
sandbox['exploratory_prediction'] = np.where(accepted, sandbox['top_sequence_id'], 'abstain')
sandbox['exploratory_correct'] = sandbox['exploratory_prediction'] == sandbox['published_sequence_id']
print('EXPLORATORY ONLY — accepted {} of {}; correct among accepted {}'.format(
    int(accepted.sum()), len(sandbox), int(sandbox.loc[accepted, 'exploratory_correct'].sum())
))
display(sandbox[['event_id', 'published_sequence_id', 'top_sequence_id', 'top_family_score', 'classification_margin', 'exploratory_prediction', 'exploratory_correct']])

## 6. What the deep DAS has established

The prospective 2026 candidate is measurable over all sampled channel blocks, and the matched same-fiber control defines a hard-negative correlation distribution. This proves detectability and supplies a preregistered control. It does not prove target-family membership, direct DAS repeatability, or source-property resolution because only one prospective candidate overlaps this deep configuration.

In [ ]:
deep_metrics = pd.read_csv(PROJECT / 'outputs' / 'deep_das' / 'event_metrics.csv')
negative = json.load(open(PROJECT / 'outputs' / 'deep_das' / 'hard_negative_baseline.json'))
display(deep_metrics[['event_id', 'role', 'status', 'peak_robust_z', 'median_channel_snr', 'detected_block_fraction', 'usable_power_snr_ranges_hz']])
display(pd.DataFrame([negative]))
display(Image(filename=str(PROJECT / 'outputs' / 'deep_das' / 'hard_negative_baseline.png')))

## 7. The experiment that can establish DAS value

The manifests below are commitments, not completed results. Network-only must be frozen first. DAS-only must not inherit network trigger times. Joint fusion sees only the two frozen score tables. Detection extension is event recall at matched event-level false-discovery rate; classification extension is event-held-out macro F1 and merge rate.

In [ ]:
pipelines = pd.read_csv(OUT / 'pipeline_manifest.csv')
metrics = pd.read_csv(OUT / 'metric_gates.csv')
claims = pd.read_csv(OUT / 'claim_status.csv')
display(pipelines)
display(metrics)
display(claims)

next_steps = pd.DataFrame([
    {'order': 1, 'deliverable': 'Best network-only baseline', 'scope': 'same DAS UTC/configuration intervals; multi-template HRSN/NCSN detection plus differential relocation', 'current': 'NOT RUN'},
    {'order': 2, 'deliverable': 'DAS-only candidate table', 'scope': 'array-coherent or multi-channel template scan blind to network triggers', 'current': 'NOT RUN'},
    {'order': 3, 'deliverable': 'Blind adjudication', 'scope': 'deduplicate candidates and assign event-level truth without pipeline identity', 'current': 'NOT RUN'},
    {'order': 4, 'deliverable': 'Joint frozen fusion', 'scope': 'combine score tables without test-label retuning', 'current': 'NOT RUN'},
    {'order': 5, 'deliverable': 'Interval bootstrap', 'scope': 'recall/FDR, macro F1, merge rate, and magnitude/noise/configuration strata', 'current': 'NOT RUN'},
])
display(next_steps)

## 8. Rebuild controls

The two switches only rebuild the completed external-label and HRSN correlation diagnostics. The large continuous network/DAS scans are intentionally not hidden inside this notebook; they require separate registered jobs and output manifests.

In [ ]:
if REBUILD_EXTERNAL_LABELS or REBUILD_HRSN_DIAGNOSTIC:
    import subprocess
    commands = []
    if REBUILD_EXTERNAL_LABELS:
        commands.append([sys.executable, '-m', 'faultzone.repeaters_v2.src.build_external_validation'])
    if REBUILD_HRSN_DIAGNOSTIC:
        commands.append([sys.executable, '-m', 'faultzone.repeaters_v2.src.run_network_family_benchmark'])
    for command in commands:
        print('RUN', ' '.join(command))
        subprocess.run(command, cwd=NOTEBOOK_ROOT, check=True)
else:
    print('Cached advisor mode: no network or raw data read.')